In [2]:
import os
print(os.getcwd())

/home/ubuntu


In [3]:
# بارگذاری داده تست و آموزش از local
import pandas as pd
from datasets import Dataset
from datasets import Image as HFImage
import os
local_data_root = "/home/ubuntu/dataset/"
csv_path = os.path.join(local_data_root, "kvasir_vqa_x1_with_paths.csv")
csv_path_test = os.path.join(local_data_root, "kvasir_vqa_x1_test_with_paths.csv")
base_image_dir = os.path.join(local_data_root, "kvasir_images_by_category/")
df = pd.read_csv(csv_path)
df_test = pd.read_csv(csv_path_test)
df['image'] = base_image_dir + df['path']
df_test['image'] = base_image_dir + df_test['path']
df['image_path'] = base_image_dir + df['path']
df_test['image_path'] = base_image_dir + df_test['path']

print("\nDataFrame head train:")
print(df.head())
print("\nDataFrame head test:")
print(df_test.head())
dataset = Dataset.from_pandas(df)
dataset_test = Dataset.from_pandas(df_test)
print("\nCasting 'image' column to Image feature...")
dataset = dataset.cast_column("image", HFImage())
print("\nCasting 'image' column test to Image feature...")
dataset_test = dataset_test.cast_column("image", HFImage())
print("\nSuccessfully loaded and accessed the first train image object:")
first_image = dataset[0]['image']
print(first_image)
print("\nSuccessfully loaded and accessed the first test image object:")
first_image_test = dataset_test[0]['image']
print(first_image_test)


DataFrame head train:
                      img_id  complexity  \
0  clb0kvxvm90y4074yf50vf5nq           3   
1  cl8k2u1r71foz083278j63qnm           2   
2  cl8k2u1qa1ekz08324rek2qcv           3   
3  cla820gmss67b071u3h7o5k3t           3   
4  clb0kvxvf90l4074y85pi02pq           1   

                                            question  \
0  Are there any abnormalities, polyps, or anatom...   
1  What procedure is depicted in the image and wh...   
2  Have all polyps been removed, is there any tex...   
3  Are there any surgical instruments, polyps, or...   
4  Are there any medical devices visible in the i...   

                                              answer  \
0  Evidence of oesophagitis is present with no po...   
1  Evidence of a colonoscopy with a paris iia pol...   
2  Polyps remain present, text is visible, and th...   
3  No surgical instruments or polyps are visible,...   
4        No foreign bodies or instruments identified   

                                      

In [4]:
#for cload train test
dataset[0]
dataset_test[0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=720x576>,
 'complexity': 1,
 'question': 'What type of polyp is observed in the gastrointestinal tract?',
 'answer': 'no polypoid lesions identified',
 'original': '[\n{\n"q": "What type of polyp is present?",\n"a": "none"\n}\n]',
 'question_class': "['polyp_type']",
 'img_id': 'clb0kvxwr92ng074yc6jndr8l',
 'source': 'Esophagitis',
 'path': 'kvasir_images_by_category/Esophagitis/clb0kvxwr92ng074yc6jndr8l.png',
 'image_path': '/home/ubuntu/dataset/kvasir_images_by_category/kvasir_images_by_category/Esophagitis/clb0kvxwr92ng074yc6jndr8l.png'}

In [5]:
# برای A100
#for cload
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.381 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [6]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, 
    finetune_language_layers   = True, 
    finetune_attention_modules = True, 
    finetune_mlp_modules       = True,

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407, # Seed, Remove likelihood function
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.visual` require gradients


In [7]:
import re

def extract_assistant_response(raw_text: str) -> str:
    match = re.search(r'assistant\s*(.*?)$', raw_text, re.IGNORECASE | re.DOTALL)
    
    if match:
        
        extracted_text = match.group(1).strip()
    else:
        
        extracted_text = raw_text
    final_answer = extracted_text.replace("<|eot_id|>", "").strip()

    return final_answer

In [8]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode as IM
import random, numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ----------------------------
# تعریف weak augmentation (اختیاری)
# ----------------------------
def weak_transform(img: Image.Image):
    return T.Compose([
        T.RandomResizedCrop(img.size[::-1], scale=(0.9,1.0),
                            ratio=(img.size[0]/img.size[1]*0.95,
                                   img.size[0]/img.size[1]*1.05),
                            interpolation=IM.BICUBIC),
        T.RandomRotation((-10,10), interpolation=IM.BICUBIC, fill=0),
        T.RandomAffine(0, translate=(0.1,0.1), interpolation=IM.BICUBIC, fill=0),
        T.ColorJitter(0.2,0.2)
    ])(img)

# ----------------------------
# Dataset سفارشی با resize و conversion در لحظه
# ----------------------------
class GastroConversationDataset(Dataset):
    def __init__(self, dataset, image_size=224, use_augmentation=False):
        self.dataset = dataset  # شامل image, question, answer
        self.image_size = image_size
        self.use_augmentation = use_augmentation

        # تعریف resize transform برای تصاویر
        self.resize = T.Resize((image_size, image_size), interpolation=IM.BICUBIC)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        question = sample["question"]
        answer = sample["answer"]
        image = sample["image"].convert("RGB")

        # Resize تصویر به 224x224
        image = self.resize(image)

        # اعمال weak augmentation در صورت نیاز
        if self.use_augmentation:
            image = weak_transform(image)

        # ساخت instruction و conversation
        instruction = f"""You are an expert gastroenterologist. Your task is to analyze the image and answer the user's question.
**Constraint:** Your entire response must be a single, concise sentence. Do not elaborate or provide additional context.
**Question:** {question}"""

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": image},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ]
        return {"messages": conversation}


In [9]:
import wandb
import os
wandb.login(key="abeb7ccc76d9074a5d2e101de45279ab0df278fb")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ubuntu/.netrc
wandb: Currently logged in as: mahdiazmoodeh95 (mahdiazmoodeh95-shahid-beheshti-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
dataset_easy = dataset.filter(lambda x: x["complexity"] == 1, num_proc=32)
train_ds_easy = GastroConversationDataset(dataset_easy, image_size=224, use_augmentation=True)

Filter (num_proc=32):   0%|          | 0/143594 [00:00<?, ? examples/s]

In [11]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
print("\n===== 🧠 Training Stage 1: EASY ONLY =====")
FastVisionModel.for_training(model)

trainer_easy = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds_easy,
    args=SFTConfig(
        per_device_train_batch_size=128,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        learning_rate=5e-5,
        num_train_epochs=1,
        logging_steps=5,
        optim="adamw_8bit",
        # weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_easy",
        report_to="wandb",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)



===== 🧠 Training Stage 1: EASY ONLY =====
Unsloth: Model does not have a default image size - using 512


In [12]:
stats_easy=trainer_easy.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,360 | Num Epochs = 1 | Total steps = 97
O^O/ \_/ \    Batch size per device = 128 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (128 x 4 x 1) = 512
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.791700
10,2.239000
15,1.699600
20,1.270400
25,0.912000
30,0.702200
35,0.577700
40,0.547300
45,0.521100
50,0.505100


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 267b7715-0268-40e2-965d-d51b23047c28)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2-VL-2B-Instruct-bnb-4bit/7a5816478cef9812cefab3303bb998c269e46965/config.json
Retrying in 1s [Retry 1/5].


In [13]:
print(stats_easy)

TrainOutput(global_step=97, training_loss=0.8179374085259192, metrics={'train_runtime': 3555.0059, 'train_samples_per_second': 13.885, 'train_steps_per_second': 0.027, 'total_flos': 9.447012025132646e+16, 'train_loss': 0.8179374085259192, 'epoch': 1.0})


In [14]:
trainer_easy.save_model("outputs_easy")
print("✅ Stage 1 completed and saved to outputs_easy")

✅ Stage 1 completed and saved to outputs_easy


In [ ]:
from huggingface_hub import HfApi, HfFolder
username = "lama4vision"
HfFolder.save_token("")


In [25]:
repo_easy = f"{username}/Qwen2-VL-gastro-easy"
api = HfApi() 
api.create_repo(repo_id=repo_easy, exist_ok=True, private=False)
api.upload_folder(folder_path="outputs_easy", repo_id=repo_easy, commit_message="Stage 1 (easy) model upload")
print(f"✅ Stage 1 pushed to https://huggingface.co/{repo_easy}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Stage 1 pushed to https://huggingface.co/lama4vision/Qwen2-VL-gastro-easy


In [18]:
del dataset_easy
del train_ds_easy

In [19]:
dataset_easy_medium = dataset.filter(lambda x: x["complexity"] in [1, 2], num_proc=32)
train_easy_medium = GastroConversationDataset(dataset_easy_medium, image_size=224, use_augmentation=True)

Filter (num_proc=32):   0%|          | 0/143594 [00:00<?, ? examples/s]

In [20]:
print("\n===== 🧠 Training Stage 2: EASY + MEDIUM =====")
FastVisionModel.for_training(model)

trainer_easy_medium = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_easy_medium,
    args=SFTConfig(
        per_device_train_batch_size=128,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        learning_rate=3e-5,
        num_train_epochs=1,
        logging_steps=5,
        optim="adamw_8bit",
        # weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_easy_medium",
        report_to="wandb",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)



===== 🧠 Training Stage 2: EASY + MEDIUM =====
Unsloth: Model does not have a default image size - using 512


In [21]:
stats_easy_medium = trainer_easy_medium.train()
print(stats_easy_medium)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 96,458 | Num Epochs = 1 | Total steps = 189
O^O/ \_/ \    Batch size per device = 128 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (128 x 4 x 1) = 512
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Step,Training Loss
5,0.611100
10,0.555200
15,0.511500
20,0.451600
25,0.400300
30,0.357900
35,0.341700
40,0.322300
45,0.314000
50,0.295400


TrainOutput(global_step=189, training_loss=0.28515128546921664, metrics={'train_runtime': 7580.5348, 'train_samples_per_second': 12.724, 'train_steps_per_second': 0.025, 'total_flos': 2.123378861031936e+17, 'train_loss': 0.28515128546921664, 'epoch': 1.0})


In [22]:
trainer_easy_medium.save_model("outputs_easy_medium")
print("✅ Stage 2 completed and saved to outputs_easy_medium")

✅ Stage 2 completed and saved to outputs_easy_medium


In [26]:
repo_easy_medium = f"{username}/Qwen2-VL-gastro-easy-medium"
api.create_repo(repo_id=repo_easy_medium, exist_ok=True, private=False)
api.upload_folder(folder_path="outputs_easy_medium", repo_id=repo_easy_medium, commit_message="Stage 2 (easy+medium) upload")
print(f"✅ Stage 2 pushed to https://huggingface.co/{repo_easy_medium}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Stage 2 pushed to https://huggingface.co/lama4vision/Qwen2-VL-gastro-easy-medium


In [27]:
train_all = GastroConversationDataset(dataset, image_size=224, use_augmentation=True)

In [28]:
print("\n===== 🧠 Training Stage 3: ALL LEVELS =====")
FastVisionModel.for_training(model)

trainer_all = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_all,
    args=SFTConfig(
        per_device_train_batch_size=128,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        learning_rate=2e-5,
        num_train_epochs=1,
        logging_steps=5,
        optim="adamw_8bit",
        # weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_all",
        report_to="wandb",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)



===== 🧠 Training Stage 3: ALL LEVELS =====
Unsloth: Model does not have a default image size - using 512


In [ ]:
stats_all = trainer_all.train()
print(stats_all)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 143,594 | Num Epochs = 1 | Total steps = 281
O^O/ \_/ \    Batch size per device = 128 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (128 x 4 x 1) = 512
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Step,Training Loss
5,0.304100
10,0.287400
15,0.282700
20,0.276800
25,0.274400
30,0.275200
35,0.268300
40,0.261300
45,0.260300
50,0.258500


In [ ]:
trainer_all.save_model("outputs_all")
print("✅ Stage 3 completed and saved to outputs_all")

In [ ]:
from huggingface_hub import HfApi, HfFolder
team_username = ""
HfFolder.save_token("")


In [ ]:
repo_all = f"{team_username}/Qwen2-VL-gastro-all"
api = HfApi() 
api.create_repo(repo_id=repo_all, exist_ok=True, private=False)
api.upload_folder(folder_path="outputs_all", repo_id=repo_all, commit_message="Stage 3 (all) upload")
print(f"✅ Stage 3 pushed to https://huggingface.co/{repo_all}")

print("\n🎯 تمام مراحل آموزش و push با موفقیت انجام شد!")

In [ ]:
from huggingface_hub import HfApi, HfFolder
username = ""
HfFolder.save_token("")


In [ ]:
repo_all = f"{username}/Qwen2-VL-gastro-all"
api = HfApi() 
api.create_repo(repo_id=repo_all, exist_ok=True, private=False)
api.upload_folder(folder_path="outputs_all", repo_id=repo_all, commit_message="Stage 3 (all) upload")
print(f"✅ Stage 3 pushed to https://huggingface.co/{repo_all}")

print("\n🎯 تمام مراحل آموزش و push با موفقیت انجام شد!")